# Interfacing: API, Frontend und MCP

Im vorherigen Notebook haben wir ein Sprachmodell dazu gebracht, JSON zu liefern, das einem Schema folgt. In diesem Notebook geht es um den nächsten Schritt: **Wie gelangt dieses JSON von einem Programm zum anderen?**

Wir verfolgen ein einziges Schema von Anfang bis Ende, das **Produkt** aus dem Katalog von Auto Quote:

1. Erinnern Sie sich an JSON? Das Schema, das unser roter Faden sein wird
2. Was ist eine API? Eine kleine API, in diesem Notebook gebaut und aufgerufen
3. Sicherheit: warum sie ins Backend gehört
4. Frontend: einfach ein weiterer Client derselben API
5. KI-Agenten: Dokumentation ist für Menschen, MCP ist für Modelle

Alles läuft in diesem Notebook: Die Server starten im Hintergrund, und wir rufen sie aus der jeweils nächsten Zelle auf.

In [ ]:
# Setup - Kann ignoriert werden
import functools
import json
import os
import secrets
import threading
import time
from pathlib import Path
from typing import Annotated
from uuid import uuid4

import requests
import uvicorn
from dotenv import load_dotenv
from fastapi import APIRouter, Depends, FastAPI, Header, HTTPException
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL = "gpt-6-luna"  # Text und Tool Calling

# Vom Arbeitsverzeichnis aufwärts bis zum Repo-Root (dem Ordner mit .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())


@functools.cache
def get_client() -> OpenAI:
    load_dotenv(REPO_ROOT / ".env.local")
    return OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)


client = get_client()

RUNNING_SERVERS: dict[int, tuple[uvicorn.Server, threading.Thread]] = {}


def serve(app, port: int) -> str:
    """Startet eine ASGI-App in einem Hintergrund-Thread und gibt ihre Basis-URL zurück. Wird die Zelle erneut ausgeführt, startet der Server neu."""
    stop(port)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        time.sleep(0.05)
    RUNNING_SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


def stop(port: int) -> None:
    if port in RUNNING_SERVERS:
        server, thread = RUNNING_SERVERS.pop(port)
        server.should_exit = True
        thread.join()


def show(response: requests.Response) -> None:
    print(response.status_code, response.reason)
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))

## 1. Erinnern Sie sich an JSON?

Im vorherigen Notebook hat ein JSON-Schema genau beschrieben, welche Keys und Typen wir vom Modell erwarten. JSON ist die gemeinsame Sprache unserer Anwendung, und Sie sehen es an zwei Stellen:

- **Gespeichert**: Die "Datenbank" des Prototyps besteht aus JSON-Dateien in `backend/data/` (zum Beispiel `products.json`, das beim ersten Start des Backends erstellt wird).
- **Unterwegs**: Jede Nachricht zwischen Frontend und Backend ist ein JSON-Payload. Darum geht es in diesem Notebook.

Unser roter Faden ist das Produkt aus dem Katalog. Es ist eine gekürzte Kopie von `ProductInput` und `Product` aus `backend/src/auto_quote/models/product.py`. Wir stellen uns einen Metallbaubetrieb vor, der Profile, Bleche und Stangen sowie Bearbeitungsdienstleistungen verkauft. Die Daten unten sind Beispieldaten.

In [ ]:
class ProductInput(BaseModel):
    product_reference: str = Field(min_length=1)
    description: str
    unit: str
    unit_price: float | None = Field(ge=0, description="Price in CHF per unit, null when the price is unknown.")
    spec: dict[str, str]


class Product(ProductInput):
    id: str
    create_timestamp: int
    update_timestamp: int


CATALOGUE = [
    {
        "product_reference": "HEA-200",
        "description": "Structural steel beam HEA 200",
        "unit": "m",
        "unit_price": 78.0,
        "spec": {"steel_grade": "S355JR", "weight": "42.3 kg/m"},
    },
    {
        "product_reference": "SS-304-2",
        "description": "Stainless steel sheet 2 mm",
        "unit": "m²",
        "unit_price": 89.0,
        "spec": {"alloy": "1.4301 (AISI 304)", "thickness": "2 mm", "finish": "2B"},
    },
    {
        "product_reference": "AL-6060-R40",
        "description": "Aluminium round bar, diameter 40 mm",
        "unit": "m",
        "unit_price": 38.5,
        "spec": {"alloy": "EN AW-6060", "diameter": "40 mm"},
    },
    {
        "product_reference": "CUT-LASER",
        "description": "Laser cutting of steel sheets",
        "unit": "hour",
        "unit_price": 145.0,
        "spec": {"max_thickness": "20 mm"},
    },
    {
        "product_reference": "WELD-MIG",
        "description": "MIG/MAG welding by a certified welder",
        "unit": "hour",
        "unit_price": 98.0,
        "spec": {"certification": "ISO 9606-1"},
    },
]

print(json.dumps(CATALOGUE[0], indent=2))

Dasselbe Produkt als JSON-Schema, das Pydantic erzeugt. Genau diese Art von Schema haben wir im vorherigen Notebook dem Modell übergeben. Hier beschreibt es, was unsere eigene API akzeptiert.

In [ ]:
print(json.dumps(ProductInput.model_json_schema(), indent=2))

## 2. Was ist eine API?

Eine **API** (Application Programming Interface) ist eine Tür in ein Programm, die andere Programme benutzen können. Statt dass eine Person auf einem Bildschirm klickt, sendet ein Programm eine Anfrage (Request) und erhält eine Antwort (Response).

```
 Client                                   Server
(Frontend, CRM, Skript, KI-Agent)        (unser Backend)
        │                                     │
        │  POST /api/v1/products + JSON-Body  │
        │ ──────────────────────────────────► │  validiert, speichert
        │                                     │
        │  200 OK  + JSON-Response            │
        │ ◄────────────────────────────────── │
```

Die meisten Web-APIs (REST) funktionieren über HTTP: Der **Client** sendet einen Request, der **Server** schickt eine Response zurück. Ein Request hat eine **Methode** (was zu tun ist), einen **Pfad** (womit), Header und optional einen **JSON-Body**. Eine Response hat einen **Statuscode**, Header und meistens einen JSON-Body.

Der Client kann unser Frontend sein, ein CRM, ein Skript oder ein KI-Agent. Für den Server spielt das keine Rolle.

| Methode | Bedeutung | Beispiel |
|---|---|---|
| `GET` | lesen | `GET /api/v1/products` |
| `POST` | erstellen | `POST /api/v1/products` mit einem Produkt als Body |
| `PUT` | ersetzen | `PUT /api/v1/products/{product_id}` mit einem Produkt als Body |
| `DELETE` | löschen | `DELETE /api/v1/products/{product_id}` |

| Status | Bedeutung |
|---|---|
| `200` | hat funktioniert |
| `401` | Sie sind nicht identifiziert |
| `404` | existiert nicht |
| `422` | der Payload entspricht nicht dem Schema |

Weil jede Programmiersprache HTTP senden und JSON parsen kann, verbindet eine API sehr unterschiedliche Systeme. Deshalb ist das JSON-Schema so wichtig: Es ist der Vertrag, auf den sich beide Seiten einigen.

### Eine kleine API

Mit [FastAPI](https://fastapi.tiangolo.com/) werden die Pydantic-Modelle von oben zum Vertrag: Ein Payload, der nicht zu `ProductInput` passt, erreicht unseren Code nie. Wir bauen dieselben drei Produkt-Endpoints wie die echte App (`backend/src/auto_quote/routes/product.py`): auflisten, erstellen und aktualisieren, unter demselben Pfad `/api/v1/products`. Als Speicher dient ein Python-Dict, das so lange lebt wie das Notebook, während Auto Quote in JSON-Dateien schreibt.


In [ ]:
PRODUCTS_PATH = "/api/v1/products"
PRODUCT_PATH = f"{PRODUCTS_PATH}/{{product_id}}"

products: dict[str, Product] = {}
router = APIRouter()


def now_timestamp() -> int:
    return time.time_ns() // 1_000_000


@router.get(PRODUCTS_PATH)
def get_products() -> list[Product]:
    return list(products.values())


@router.post(PRODUCTS_PATH)
def create_product(product_input: ProductInput) -> Product:
    timestamp = now_timestamp()
    product = Product(id=str(uuid4()), create_timestamp=timestamp, update_timestamp=timestamp, **product_input.model_dump())
    products[product.id] = product
    return product


@router.put(PRODUCT_PATH)
def update_product(product_id: str, product_input: ProductInput) -> Product:
    if product_id not in products:
        raise HTTPException(status_code=404, detail=f"Product {product_id} not found")
    product = products[product_id].model_copy(update={**product_input.model_dump(), "update_timestamp": now_timestamp()})
    products[product_id] = product
    return product


app = FastAPI(title="Products")
app.include_router(router)

BASE_URL = serve(app, port=8001)
print("API läuft auf", BASE_URL)


Die Zelle ist fertig, aber der Server läuft weiter: Er läuft in einem **Hintergrund-Thread** dieses Notebooks und wartet auf Requests. Sie können also einfach die nächsten Zellen ausführen, die ihn aufrufen.

- Wird die Zelle oben erneut ausgeführt, startet der Server neu, mit leerem Katalog.
- Zum Stoppen: Führen Sie `stop(8001)` in einer Zelle aus, oder die letzte Zelle des Notebooks, die alle hier gestarteten Server stoppt. Auch ein Neustart oder das Schliessen des Kernels stoppt sie.

Jetzt sind wir der Client. Zuerst ein leerer Katalog:


In [ ]:
show(requests.get(f"{BASE_URL}{PRODUCTS_PATH}"))

Wir senden unseren Katalog, ein JSON-Payload pro Produkt. `json=` serialisiert das Dict und setzt den Header `Content-Type: application/json`.

In [ ]:
for product_payload in CATALOGUE:
    response = requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=product_payload)
    print(response.status_code, response.json()["id"], response.json()["product_reference"])

`PUT` ersetzt ein Produkt: Wir senden das vollständige Produkt mit einem neuen Preis an die URL seiner `id`. Beachten Sie, dass sich `update_timestamp` ändert und `create_timestamp` nicht.

In [ ]:
hea_200 = next(product for product in requests.get(f"{BASE_URL}{PRODUCTS_PATH}").json() if product["product_reference"] == "HEA-200")
price_update = {**CATALOGUE[0], "unit_price": 82.0}
show(requests.put(f"{BASE_URL}{PRODUCTS_PATH}/{hea_200['id']}", json=price_update))

Ein Produkt zu aktualisieren, das nicht existiert, ergibt einen `404`. In Auto Quote ist das der `NotFoundError` des Controllers, der am Rand der App in einen Statuscode übersetzt wird (`backend/src/auto_quote/server.py`).

In [ ]:
show(requests.put(f"{BASE_URL}{PRODUCTS_PATH}/DOES-NOT-EXIST", json=CATALOGUE[0]))

Ein Payload, der das Schema verletzt, wird abgelehnt, bevor unser Code läuft. Hier ist der Preis negativ und `spec` fehlt. Die Antwort sagt genau, was falsch ist. Das hilft Entwicklerinnen und Entwicklern und ebenso einem KI-Agenten.

In [ ]:
bad_payload = {"product_reference": "XX-1", "description": "Broken", "unit": "piece", "unit_price": -5}
show(requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=bad_payload))

## 3. Sicherheit

Bisher kann jeder, der den Server erreicht, unseren Katalog lesen und ändern. Dieser Abschnitt soll Sicherheit nicht umfassend behandeln, das ist ein eigenes Thema, sondern einen Punkt klarmachen:

> **Sicherheit gehört ins Backend.**

Ein Frontend kann einen Button oder eine Seite ausblenden, aber das hilft nur ehrlichen Nutzerinnen und Nutzern. Die API ist auch ohne Frontend erreichbar, genau so, wie wir es eben aus Python gemacht haben. Wer sie missbrauchen will, benutzt nicht unsere Oberfläche. Deshalb muss jeder Endpoint auf dem Server prüfen, **wer aufruft** und **was diese Person darf**.

Unten die einfachste Form: Der Aufrufer muss einen geheimen Schlüssel in einem Header mitsenden. Wir verwenden denselben Router und dieselben Daten und setzen nur eine Prüfung vor jeden Endpoint.

In [ ]:
API_KEY = "demo-key"  # Nur für die Demo: Ein echter Schlüssel wird aus der Umgebung gelesen und nie in den Code geschrieben


def require_api_key(x_api_key: Annotated[str | None, Header()] = None) -> None:
    if x_api_key is None or not secrets.compare_digest(x_api_key.encode(), API_KEY.encode()):
        raise HTTPException(status_code=401, detail="Missing or invalid API key")


secured_app = FastAPI(title="Products (secured)")
secured_app.include_router(router, dependencies=[Depends(require_api_key)])

SECURED_URL = serve(secured_app, port=8002)
print("Gesicherte API läuft auf", SECURED_URL)

In [ ]:
print("Ohne Schlüssel:")
show(requests.get(f"{SECURED_URL}{PRODUCTS_PATH}"))

print("\nFalscher Schlüssel:")
show(requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers={"X-API-Key": "guess"}))

In [ ]:
API_HEADERS = {"X-API-Key": API_KEY}
response = requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers=API_HEADERS)
print(response.status_code, [p["product_reference"] for p in response.json()])

In Auto Quote deklariert jede Route den Benutzer, für den sie arbeitet, und FastAPI führt die Prüfung vor dem Code der Route aus. Aus `backend/src/auto_quote/routes/product.py`:

```python
CurrentUser = Annotated[User, Depends(auth_provider.get_current_user)]

@router.get(ApiPaths.PRODUCTS)
async def get_products(user: CurrentUser) -> list[Product]:
    return await product_controller.get_products()
```

Das ist derselbe Mechanismus wie unser `Depends(require_api_key)`. Der Prototyp identifiziert Benutzer mit einem Session-Cookie (`auth/session_cookie.py`); ein produktives System würde hinter demselben `AuthProvider` ein echtes Login anschliessen.

Daraus folgen drei Gewohnheiten:

- Identität und Berechtigungen auf dem Server prüfen, bei jedem Endpoint.
- Nie ein Secret ins Frontend oder in ein Notebook schreiben, das Sie teilen: Alles, was der Browser erhält, kann der Benutzer lesen.
- Den Input auf dem Server validieren. Das Schema oben ist genau diese Validierung.

## 4. Frontend

Das Frontend von Auto Quote (React, in `frontend/`) ist **ein Client der API**, genau wie unsere `requests`-Aufrufe. Es hat keinen direkten Zugriff auf die Daten: Wenn Sie die Katalogseite öffnen, sendet der Browser `GET /api/v1/products` und stellt das zurückgegebene JSON dar. Wenn Sie ein Produkt speichern, sendet er einen `POST` mit einem JSON-Payload.

Hier ist die Datei, die das für Produkte erledigt. Sie hat dieselben drei Aufrufe wie unsere kleine API, und das TypeScript-Interface `ProductInput` spiegelt unser Pydantic-Modell: Es ist derselbe Vertrag, zweimal geschrieben.

In [ ]:
print((REPO_ROOT / "frontend" / "src" / "services" / "product.ts").read_text())

Unsere Aufrufe von oben im Vergleich:

| | Frontend (TypeScript) | Dieses Notebook (Python) |
|---|---|---|
| auflisten | `this.get<Product[]>(PRODUCTS_PATH)` | `requests.get(f"{BASE_URL}{PRODUCTS_PATH}")` |
| erstellen | `this.post<Product>(PRODUCTS_PATH, productInput)` | `requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=payload)` |
| aktualisieren | ``this.put<Product>(`${PRODUCTS_PATH}/${productId}`, productInput)`` | `requests.put(f"{BASE_URL}{PRODUCTS_PATH}/{product_id}", json=payload)` |

Sie können das live beobachten: Starten Sie die App, öffnen Sie die Entwicklertools des Browsers (**F12**), wechseln Sie in den Tab **Network** und benutzen Sie die Katalogseite. Jeder Klick zeigt einen Request mit seinem JSON-Payload und die JSON-Response.

Zwei Konsequenzen für das Design:

- Die API ist das Produkt, der Bildschirm ist eine Art, es zu benutzen. Ein zweiter Client (eine CRM-Integration, ein Skript, ein Agent) braucht kein zweites Backend.
- Das Frontend kann das Formular zur Bequemlichkeit validieren, aber entscheiden tut nur das Backend, wie wir im Abschnitt Sicherheit gesehen haben.

## 5. Der KI unsere API zugänglich machen

### Dokumentation: für Menschen gemacht

Wie erfährt eine Entwicklerin, was eine API anbietet? Sie liest die Dokumentation, und FastAPI erzeugt diese aus unserem Code. Öffnen Sie <http://127.0.0.1:8001/docs> im Browser (die kleine API aus Abschnitt 2 läuft noch): Das ist eine interaktive Seite, auf der Sie jeden Endpoint und seinen Payload nachlesen und ausprobieren können. Für Auto Quote finden Sie dieselbe Seite unter <http://localhost:8000/docs>, wenn das Backend läuft.

Hinter dieser Seite steckt eine maschinenlesbare Datei, das **OpenAPI**-Dokument. Es ist JSON, und die Schemas darin sind die JSON-Schemas, die wir oben gesehen haben.


In [ ]:
openapi = requests.get(f"{BASE_URL}/openapi.json").json()
print(list(openapi["paths"]))
print(json.dumps(openapi["components"]["schemas"]["ProductInput"], indent=2))

Das OpenAPI-Dokument ist vollständig, aber es ist für einen Menschen geschrieben, der danach den Code schreibt: Jemand entscheidet, welcher Aufruf welches Bedürfnis abdeckt, und kümmert sich um URLs, Header und Fehler. Ein KI-Modell kann keine Dokumentationsseite öffnen und auf *Try it out* klicken. Es braucht etwas anderes: eine Liste von Aktionen, die es anfordern kann, jede mit einem Namen, einer Beschreibung in Alltagssprache und einem Input-Schema, in einem Standardformat, das jeder Agent versteht.


### MCP: dieselbe Tür, gemacht für Modelle

Kann ein KI-Agent unsere API benutzen? Ja, mit der richtigen Tür. **MCP** (Model Context Protocol) ist ein offener Standard genau dafür. Ein MCP-Server stellt eine Liste von **Tools** bereit. Jedes Tool hat:

- einen **Namen**,
- eine **Beschreibung** in Alltagssprache (diese liest das Modell, um zu entscheiden, wann es das Tool benutzt),
- ein **Input-Schema**, wiederum ein JSON-Schema.

```
 Agent (LLM + MCP-Client)               MCP-Server                    unsere API
        │  list_tools                       │                              │
        │ ────────────────────────────────► │                              │
        │ ◄──── Namen, Beschreibungen, Schemas                             │
        │  call_tool get_product {...}      │  GET /products/HEA-200 + Key │
        │ ───────────────────────────────────────────────────────────────► │
        │ ◄──────────────── Resultat ───────────────────────────────────── │
```

Der MCP-Server ist eine dünne Schicht vor der API, die wir bereits haben. Der API-Schlüssel bleibt im MCP-Server: Der Agent sieht ihn nie.

### Ein kleiner MCP-Server

Wir verwenden [FastMCP](https://gofastmcp.com), eine Abhängigkeit des Backends. Wir stellen zwei Tools bereit, die nur lesen. Die Docstrings sind keine Dekoration: Sie sind die Anweisungen, die das Modell liest.

In [ ]:
from fastmcp import Client, FastMCP

mcp_server = FastMCP("metal-catalogue")


def fetch_products() -> list[dict]:
    response = requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers=API_HEADERS)
    response.raise_for_status()
    return response.json()


@mcp_server.tool
def list_products() -> list[dict]:
    """List every product of the catalogue with its id, reference, description, unit, unit price in CHF and specifications."""
    return fetch_products()


@mcp_server.tool
def get_product(product_reference: str) -> dict:
    """Get one product of the catalogue by its reference, for example HEA-200."""
    # Die API hat keine Suche nach Referenz: Ein Tool kann sich an einer Aufgabe orientieren statt an einem Endpoint.
    for product in fetch_products():
        if product["product_reference"] == product_reference:
            return product
    raise ValueError(f"No product with reference {product_reference}")


MCP_URL = serve(mcp_server.http_app(), port=8003) + "/mcp"
print("MCP-Server läuft auf", MCP_URL)


Jetzt sind wir ein MCP-Client. Als Erstes fragt ein Client, welche Tools es gibt. Achten Sie auf `input_schema`: Es ist dieselbe Art von JSON-Schema wie überall sonst in diesem Notebook.

In [ ]:
async def list_mcp_tools():
    async with Client(MCP_URL) as mcp_client:
        return await mcp_client.list_tools()


for tool in await list_mcp_tools():
    print(tool.name)
    print(" ", tool.description)
    print(" ", json.dumps(tool.input_schema))

### Den Server mit einem Agenten verbinden

Ein Agent ist eine Schleife um das Modell:

1. Die Frage zusammen mit den Tool-Beschreibungen an das Modell senden.
2. Antwortet das Modell mit einem **Tool Call**, diesen auf dem MCP-Server ausführen, das Resultat zurücksenden und zurück zu 1.
3. Antwortet das Modell mit Text, sind wir fertig.

Das Modell ruft selbst nie etwas auf: Es fordert nur einen Aufruf an, und unser Code entscheidet, ob er ausgeführt wird. Die Tool-Schemas aus MCP gehen direkt in den Parameter `tools`, weil beide Seiten JSON-Schema sprechen.

In [ ]:
def to_openai_tool(tool) -> dict:
    return {"type": "function", "name": tool.name, "description": tool.description, "parameters": tool.input_schema}


async def ask_agent(question: str, max_steps: int = 5) -> str:
    async with Client(MCP_URL) as mcp_client:
        tools = [to_openai_tool(tool) for tool in await mcp_client.list_tools()]
        messages = [{"role": "user", "content": question}]

        for _ in range(max_steps):
            response = client.responses.create(model=MODEL, input=messages, tools=tools)
            messages += response.output
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text

            for call in tool_calls:
                arguments = json.loads(call.arguments)
                print(f"tool call: {call.name}({arguments})")
                result = await mcp_client.call_tool(call.name, arguments, raise_on_error=False)
                output = "\n".join(block.text for block in result.content)
                messages.append({"type": "function_call_output", "call_id": call.call_id, "output": output})

    raise RuntimeError(f"Keine Antwort nach {max_steps} Schritten")

In [ ]:
print(await ask_agent("Wie viel kostet ein Meter HEA-200 und welche Stahlsorte ist es?"))

In [ ]:
print(await ask_agent("Welche unserer Produkte werden pro Stunde verkauft, und was würden 6 Stunden Schweissen kosten?"))

Schauen Sie sich die ausgegebenen Tool Calls an: Das Modell hat das Tool gewählt und die Argumente anhand des Schemas ausgefüllt, und unsere API hat mit demselben JSON geantwortet wie vorher. Wir haben keinen Code geschrieben, der eine Frage mit einem Endpoint verknüpft.

Derselbe MCP-Server funktioniert mit jedem MCP-Client, nicht nur mit unserer Schleife. Solange das Notebook läuft, kann sich Claude Code damit verbinden:

```sh
claude mcp add --transport http metal-catalogue http://127.0.0.1:8003/mcp
```

Zwei Dinge, die Sie beachten sollten, wenn Sie einem Agenten Zugriff auf Ihre API geben:

- **Beginnen Sie nur lesend.** Jedes Tool, das Sie hinzufügen, kann der Agent selbständig benutzen. Ein Tool, das schreibt oder löscht, verdient viel mehr Sorgfalt als die beiden oben.
- **Die Tools sind Ihre Sicherheitsgrenze.** Der Agent kann nur tun, was der MCP-Server bereitstellt, und die API prüft weiterhin bei jedem Request den Schlüssel.

## Fazit

- JSON ist das, was zwischen unseren Programmen unterwegs ist, und ein JSON-Schema ist der Vertrag, auf den sich beide Seiten einigen.
- Eine API erlaubt jedem Programm (einem Frontend, einem CRM, einem Skript), unser Backend über Requests und JSON-Payloads zu benutzen. FastAPI macht aus einem Pydantic-Modell Validierung und Dokumentation.
- Sicherheit gehört ins Backend, auf jeden Endpoint. Etwas im Frontend zu verstecken ist keine Sicherheit.
- Das Frontend ist ein Client der API; es braucht keinen eigenen Zugriff auf die Daten.
- Dokumentation ist für Menschen. MCP erlaubt einem KI-Agenten, unsere API zu entdecken und aufzurufen: Tools mit einem Namen, einer Beschreibung und einem Input-Schema. Es ist eine dünne Schicht über der API, die wir bereits haben.

### Wo Sie die einzelnen Teile in Auto Quote sehen und ändern

Starten Sie die App (`make dev` in `backend/`, `npm run dev` in `frontend/`) und betrachten Sie sie aus jedem Blickwinkel. Die Dateilinks sind relativ zu diesem Notebook.

| Teil | Beobachten | Ändern |
|---|---|---|
| **JSON und Schema** | Der Katalog wird in `backend/data/products.json` gespeichert (beim ersten Start erstellt). Das Produktschema ist [`models/product.py`](../src/auto_quote/models/product.py). | In der App unter [Admin → Documents](http://localhost:3000/admin/document-retrieval), Tabelle *Products*: Fügen Sie ein Produkt hinzu oder bearbeiten Sie eines, und beobachten Sie, wie sich `products.json` ändert. Um ein Feld hinzuzufügen, ändern Sie `ProductInput` in `models/product.py` und das passende Interface in [`services/product.ts`](../../frontend/src/services/product.ts). |
| **API** | [http://localhost:8000/docs](http://localhost:8000/docs): jede Route der App, mit *Try it out*. Die Produkt-Routen sind in [`routes/product.py`](../src/auto_quote/routes/product.py), die Pfade in [`routes/paths.py`](../src/auto_quote/routes/paths.py). | Fügen Sie eine Route zu `routes/product.py` hinzu (zum Beispiel ein `GET` nach id, das die App noch nicht hat) und die zugehörige Controller-Methode in [`controllers/product.py`](../src/auto_quote/controllers/product.py). |
| **Sicherheit** | In den Entwicklertools des Browsers (**F12**), Tab *Application*, identifiziert das Cookie `auto_quote_session` den Aufrufer. Die Prüfung ist [`auth/session_cookie.py`](../src/auto_quote/auth/session_cookie.py), hinter dem Interface in [`auth/auth_provider.py`](../src/auto_quote/auth/auth_provider.py). | In [`main.py`](../src/auto_quote/main.py) ist die Zeile `auth_provider = SessionCookieProvider(...)` die einzige Stelle, die die Implementierung wählt. Ein neuer Provider, zum Beispiel ein API-Schlüssel wie in diesem Notebook, wird dort eingesetzt. |
| **Frontend** | **F12**, Tab *Network*, dann [Admin → Documents](http://localhost:3000/admin/document-retrieval) öffnen und ein Produkt speichern: Sie sehen den Request `PUT /api/v1/products/{id}` mit seinem JSON-Payload. | Die Aufrufe sind in [`services/product.ts`](../../frontend/src/services/product.ts). Die Tabelle und der Bearbeitungsdialog sind [`products-table.tsx`](../../frontend/src/components/admin/products-table.tsx) und [`product-dialog.tsx`](../../frontend/src/components/admin/product-dialog.tsx). |
| **MCP** | In der App gibt es noch keinen MCP-Server. Der kleine Server oben zeigt, wie er aussieht, und `prototyping_21_api_to_mcp_optional_deep_dive.ipynb` wandelt die echte App um. | Die Namen und Beschreibungen der Tools sind das, was das Modell liest: Ändern Sie im Notebook einen Docstring und stellen Sie dem Agenten dieselbe Frage noch einmal. |


## Herunterfahren

Dies stoppt die drei in diesem Notebook gestarteten Server (Ports 8001, 8002 und 8003). Um nur einen zu stoppen, führen Sie zum Beispiel `stop(8001)` aus.

In [ ]:
# Stoppt die drei in diesem Notebook gestarteten Server
for port in list(RUNNING_SERVERS):
    stop(port)